# A simple RAG system for the KBC Annual Report
**Author:** Lakshya Katariya  
**Date:** 2026-10-07

This is an independent learning project that builds a small retrieval-augmented generation (RAG) system over a publicly available KBC Group annual report. It is not affiliated with or endorsed by KBC. The report is downloaded from KBC's public website and is not stored in this repository.

To install the required packages: `pip install -r requirements.txt`

## 0. Setup

In [1]:
import hashlib
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pdfplumber
import requests
import semchunk
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer

In [2]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

REPORT_URL = "https://wcmassets.kbc.be/content/dam/kbccom/doc/investor-relations/Results/jvs-2025/jvs-2025-gr-en.pdf"
REPORT_PATH = DATA_DIR / "kbc_annual_report.pdf"

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
MODEL_MAX_TOKENS = 256   # the model truncates input beyond this length
CHUNK_SIZE = 240         # in model tokens, leaving room for special tokens
MIN_CHUNK_CHARS = 100    # shorter chunks are dropped (see section 2.3)

load_dotenv()
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
assert OPENROUTER_API_KEY, "OPENROUTER_API_KEY not found. Check your .env file."
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

# Free models on OpenRouter compared in section 6 (short name: model ID)
LLM_MODELS = {
    "nemotron": "nvidia/nemotron-3-super-120b-a12b:free",
    "apodex": "apodex/apodex-1.1-mini:free",
}
DEFAULT_LLM = "nemotron"
TOP_K = 5                # number of chunks retrieved per question
PAUSE_SECONDS = 2        # pause between API calls, to respect free-tier rate limits

In [3]:
def clean_text(text):
    """Collapse all whitespace (spaces, line breaks, form feeds) into single spaces."""
    if not isinstance(text, str):
        return ""
    return " ".join(text.split())

Loading the embedding model. It is downloaded from Hugging Face on the first run and cached afterwards. Its tokenizer is also used for chunking, so chunk sizes are measured in the same units the model reads.

In [4]:
model = SentenceTransformer(EMBEDDING_MODEL)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 1. Preparing the annual report data
### 1.1 Downloading the PDF
The file is downloaded only if it is not already present in `data/`.

In [5]:
if REPORT_PATH.exists():
    print(f"Using existing file: {REPORT_PATH}")
else:
    response = requests.get(REPORT_URL, timeout=60)
    response.raise_for_status()
    REPORT_PATH.write_bytes(response.content)
    print(f"Downloaded report to {REPORT_PATH}")

Using existing file: data\kbc_annual_report.pdf


### 1.2 Extracting text page by page
Each page is stored with its PDF page number (counting from 1), which is used later for citations and evaluation. The extracted text keeps its line breaks, since the chunker uses them to find paragraph boundaries.

In [6]:
pages = []
with pdfplumber.open(REPORT_PATH) as pdf:
    for i, page in enumerate(pdf.pages, start=1):
        pages.append({"page": i, "text": page.extract_text() or ""})

pages_df = pd.DataFrame(pages)
pages_df["text_len"] = pages_df["text"].str.len()

print(f"Extracted {len(pages_df)} pages")
pages_df["text_len"].describe()

Extracted 432 pages


count      432.000000
mean      4023.460648
std       2148.574473
min          0.000000
25%       2614.250000
50%       3798.000000
75%       4998.000000
max      12331.000000
Name: text_len, dtype: float64

Inspecting the full text of one page:

In [7]:
with pd.option_context("display.max_colwidth", None):
    display(pages_df.iloc[[1]])

,page,text,text_len
1,2,"This is a Word proxy of theA annnunal urepaortl. TRhe epropfessoio-rt\nnal layout will include the correct word divisions.\nNote: some draft pictures/schemes are still in Dutch.\nKBC Group\nKBC group passport\nOur area of operation KBC group in 2025\nWe are an integrated bank-insurance group, catering mainly for retail,\n6 million clients\nprivate banking, SME and mid-cap clients. Our core markets are Belgium, have already\nused Kate; 3,6\nthe Czech Republic, Slovakia, Hungary and Bulgaria. We are present to a the number of Solid capital\nlimited extent in several other countries. cases resolved billion EUR and liquidity\nautonomously in net profit ratios\nby Kate\nOur goal and ambition continues to rise\nThrough our activities, we want to help our clients to both realise and\nChallenging Increase in\nprotect their dreams and projects. geopolitical Update of number of\ndividend policy\nIt is our ambition to be the reference for bank-insurance in all our core and independent\nand capital\neconomic directors on\nmarkets. allocation\nconditions the Board\npolicy\nof Directors\nOur clients, staff and network as at 31-12-2025\nClients approximately 13 million Recognition Acquisition of\nfrom several Advancing\n365.bank\nStaff approximately 40 000 external entities our strategy\nin Slovakia\nfor our with Save Time\nBank branches 1 090 digitalisation and Business and Earn\nInsurance network 270 agencies in Belgium, various distribution and Lease in the Money (S.T.E.M.):\nsustainability Czech Republic the Ecosphere\nchannels in Central and Eastern Europe strategy and Slovakia\nOur ratings as at 31-12-2025\nLong-term debt\nratings Fitch Moody’s S&P’s Net result\n(in millions of EUR)\nKBC Bank NV A+ A1 A+\n3 568\nKBC Insurance NV – – A 3 402 3 415\nKBC Group NV A A3 A-\n2 818\nSustainability ratings\n(selection) CDP Sustainalytics S&P Global\nKBC Group A 10.2 73/100\n1 440\n4.1 million clients\n14 000 employees\n424 13 b 1 a b n ill k io b n r a EU nc R h es 1 4 2 . 0 3 0 m 0 il l e io m n p c lo lie y n e t e s s 0.8 million clients 2022 2023 2024 2025\nin loans and 197 bank branches 3 000 employees\nadvances 44 billion EUR 97 bank branches\n148 billion EUR in loans 13 billion EUR\nin deposits and advances in loans\n54 billion EUR and advances Breakdown of net result by business unit\nin deposits 9 billion EUR\nin deposits (2025, in millions of EUR)\n2 151\nBelgium\nCzech Republic\nSlovakia\nHungary\n1.7 million clients\n4 000 employees\n191 bank branches Bulgaria 922\n8 billion EUR 814\nin loans\nand advances 2.2 million clients\n11 billion EUR 6 000 employees\nin deposits 170 bank branches\n13 billion EUR\nin loans -318\nand advances\nClients: estimates. Loans and advances: 16 billion EUR\nloans and advances to customers excluding reverse repos. in deposits\nDeposits: deposits from customers excluding debt securities and repos.\nBelgium Czech International Group\nBelgium: loans and deposits including the limited network of KBC Bank branches abroad.\nBusiness Republic Markets Centre\nUnit Business Business\nUnit Unit",2981


### 1.3 Removing empty pages

In [8]:
pages_nonempty = pages_df[pages_df["text_len"] > 0].copy().reset_index(drop=True)

print(f"Removed {len(pages_df) - len(pages_nonempty)} empty page(s); {len(pages_nonempty)} remain")

Removed 1 empty page(s); 431 remain


## 2. Chunking
### 2.1 Splitting pages into chunks
Each page is chunked separately with `semchunk`, so every chunk keeps its page number. Whitespace is normalised after chunking, because the chunker needs the original line breaks.

Note: the tokenizer may print a warning that a sequence is longer than 256 tokens. This refers to the full page being measured during chunking, not to the final chunks, which are checked in 2.2.

In [9]:
chunker = semchunk.chunkerify(model.tokenizer, chunk_size=CHUNK_SIZE)

chunks = []
for _, row in pages_nonempty.iterrows():
    for chunk in chunker(row["text"]):
        chunks.append({"page": row["page"], "chunk_text": clean_text(chunk)})

chunks_df = pd.DataFrame(chunks)
chunks_df["chunk_len"] = chunks_df["chunk_text"].str.len()

print(f"Created {len(chunks_df)} chunks from {len(pages_nonempty)} pages")
chunks_df.head()

Token indices sequence length is longer than the specified maximum sequence length for this model (357 > 256). Running this sequence through the model will result in indexing errors


Created 2165 chunks from 431 pages


,page,chunk_text,chunk_len
0,1,Annual Report KBC Group 2025,28
1,2,This is a Word proxy of theA annnunal urepaort...,1112
2,2,Clients approximately 13 million Recognition A...,770
3,2,424 13 b 1 a b n ill k io b n r a EU nc R h es...,1031
4,2,Business Republic Markets Centre Unit Business...,65


### 2.2 Checking chunk lengths against the model limit

In [10]:
n_tokens = chunks_df["chunk_text"].apply(lambda t: len(model.tokenizer.tokenize(t)))

print(f"Longest chunk: {n_tokens.max()} tokens")
print(f"Chunks over the {MODEL_MAX_TOKENS}-token limit: {(n_tokens > MODEL_MAX_TOKENS).sum()}")
chunks_df["chunk_len"].describe()

Longest chunk: 240 tokens
Chunks over the 256-token limit: 0


count    2165.000000
mean      802.032794
std       359.096413
min         3.000000
25%       464.000000
50%       873.000000
75%      1128.000000
max      1520.000000
Name: chunk_len, dtype: float64

### 2.3 Removing very short chunks
Some chunks are only a few characters long. Inspecting them shows they are mostly page footers (e.g. "22 Annual Report KBC Group 2025"), section labels, and fragments of tables, which carry little retrievable information. Chunks shorter than `MIN_CHUNK_CHARS` characters are dropped.

In [11]:
chunks_df[chunks_df["chunk_len"] < MIN_CHUNK_CHARS].sort_values("chunk_len").head(15)

,page,chunk_text,chunk_len
2121,424,473,3
1515,370,Additional information,22
1898,406,stessa RAG latoT 23 653,23
0,1,Annual Report KBC Group 2025,28
1826,402,%0.0 stessa RAG latoT 23 253,28
1802,401,% 0.0 stessa RAG latoT 23 053,29
1874,405,% 0.0 stessa RAG latoT 23 453,29
78,24,22 Annual Report KBC Group 2025,31
945,265,Annual Report KBC Group 2025 263,32
1016,276,274 Annual Report KBC Group 2025,32


In [12]:
chunks_kept = chunks_df[chunks_df["chunk_len"] >= MIN_CHUNK_CHARS].copy().reset_index(drop=True)

print(f"Removed {len(chunks_df) - len(chunks_kept)} short chunks; {len(chunks_kept)} remain")

Removed 48 short chunks; 2117 remain


## 3. Embeddings
### 3.1 Embedding all chunks
Embeddings are normalised to length 1, so cosine similarity equals the dot product.

They are cached in `data/`. The cache file name contains a fingerprint of the model name and the exact chunk texts, so the embeddings are recomputed automatically whenever the chunks change (for example after changing `CHUNK_SIZE` or the cleaning steps), and a stale cache is never reused by mistake.

In [13]:
fingerprint = hashlib.sha256(
    (EMBEDDING_MODEL + "\n".join(chunks_kept["chunk_text"])).encode("utf-8")
).hexdigest()[:12]
embeddings_path = DATA_DIR / f"embeddings_{fingerprint}.npy"

if embeddings_path.exists():
    embeddings = np.load(embeddings_path)
    print(f"Loaded cached embeddings from {embeddings_path}")
else:
    embeddings = model.encode(
        chunks_kept["chunk_text"].tolist(),
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    np.save(embeddings_path, embeddings)
    print(f"Computed and saved embeddings to {embeddings_path}")

print(embeddings.shape)  # (number of chunks, 384)

Loaded cached embeddings from data\embeddings_64c836c28e23.npy
(2117, 384)


## 4. Retrieval
A question is embedded with the same model as the chunks. Because all vectors are normalised, the dot product with every chunk embedding gives the cosine similarity, and the top `k` chunks are returned.

In [14]:
def retrieve(query, k=TOP_K):
    """Return the k chunks most similar to the query, with page numbers and scores."""
    query_emb = model.encode([query], normalize_embeddings=True)[0]
    scores = embeddings @ query_emb  # cosine similarity, since all vectors are normalised
    top_idx = np.argsort(scores)[::-1][:k]

    results = chunks_kept.iloc[top_idx][["page", "chunk_text"]].copy()
    results["score"] = scores[top_idx]
    return results.reset_index(drop=True)

Example: the top chunks retrieved for a question about climate policy.

In [15]:
with pd.option_context("display.max_colwidth", 200):
    display(retrieve("What is the climate change policy at KBC?"))

,page,chunk_text,score
0,156,"adjusting our processes, policies, and portfolios in order to be prepared for possible (disrupting) medium- or long-term climate change impacts on capital and as such avoid severe future impacts c...",0.704295
1,175,Climate change (2.2) Climate change strategy (2.2.1) Transition plan for climate change mitigation (2.2.1.1) [E1-1] KBC’s transition plan encompasses our clear and actionable strategy for climate ...,0.694960
2,281,assumptions that KBC’s management believes are reasonable at the time the financial statements are being prepared. which we invest in order to reduce their climate impact. Revisions to accounting ...,0.685043
3,179,Reference to third-party agreements This policy is based on the Greenhouse Gas Protocol. Areas addressed: Climate change mitigation Yes This policy addresses the restatement of our external climat...,0.668750
4,181,Exclusion Policy for Responsible Investing Funds Scope This policy is applicable to all Responsible Investing funds managed by (all subsidiaries of) KBC Asset Management. Most senior level account...,0.658112


## 5. Generation
### 5.1 Calling an LLM through OpenRouter
One function serves all models; the model is chosen by its short name from `LLM_MODELS`. `temperature=0` keeps answers as consistent as possible between runs, which matters for comparing models.

The parameter is called `llm` rather than `model`, to avoid confusion with the embedding model loaded in section 0.

In [16]:
def call_llm(messages, llm=DEFAULT_LLM, temperature=0):
    """Send a chat request to OpenRouter and return the model's reply text.

    `llm` is a short name from LLM_MODELS (e.g. "nemotron").
    """
    response = requests.post(
        OPENROUTER_URL,
        headers={"Authorization": f"Bearer {OPENROUTER_API_KEY}"},
        json={"model": LLM_MODELS[llm], "messages": messages, "temperature": temperature},
        timeout=120,
    )
    if response.status_code != 200:
        print(response.text)  # shows OpenRouter's error message
        response.raise_for_status()
    content = response.json()["choices"][0]["message"]["content"]
    return (content or "").strip()

Connection test for each model:

In [17]:
for name in LLM_MODELS:
    reply = call_llm([{"role": "user", "content": "Reply with exactly: connection works"}], llm=name)
    print(f"{name}: {reply}")
    time.sleep(PAUSE_SECONDS)

nemotron: connection works
apodex: connection works


### 5.2 Prompt
The system prompt keeps the model grounded in the retrieved excerpts, requires page citations so answers can be checked against the PDF, and fixes the wording of a refusal so that "does not know" answers can be detected automatically during evaluation.

In [18]:
SYSTEM_PROMPT = """You answer questions about the KBC Group annual report.
Use ONLY the report excerpts provided. Do not use outside knowledge.
Cite the page of every fact you use, in the form [p. 12].
If the excerpts do not contain the answer, reply exactly:
"The provided excerpts do not contain this information."
Some excerpts may contain text extraction errors; ignore fragments that are unreadable.
Keep answers short and factual."""

In [19]:
def build_messages(question, retrieved):
    """Format retrieved chunks as labelled excerpts and combine them with the question."""
    context = "\n\n".join(
        f"[Page {row.page}]\n{row.chunk_text}" for row in retrieved.itertuples()
    )
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Report excerpts:\n\n{context}\n\nQuestion: {question}"},
    ]

### 5.3 Answering a question
`answer` retrieves the chunks and generates a reply. It also returns the retrieved pages, so a wrong answer can be traced to either retrieval (the right page was not found) or generation (the right page was found but misused).

In [20]:
def answer(question, llm=DEFAULT_LLM, k=TOP_K):
    """Retrieve relevant chunks and generate an answer grounded in them."""
    retrieved = retrieve(question, k=k)
    return {
        "question": question,
        "llm": llm,
        "answer": call_llm(build_messages(question, retrieved), llm=llm),
        "retrieved_pages": retrieved["page"].tolist(),
        "retrieved": retrieved,
    }


def show(result):
    """Print a question, its answer, and the pages that were retrieved."""
    print(f"Q: {result['question']}  (model: {result['llm']})\n")
    print(f"A: {result['answer']}\n")
    print(f"Retrieved pages: {result['retrieved_pages']}")

In [21]:
show(answer("What is the climate change policy at KBC?"))

Q: What is the climate change policy at KBC?  (model: nemotron)

A: KBC’s climate change policy includes a transition plan for mitigation aligned with the Paris Agreement (well‑below 2 °C, pursuing 1.5 °C) with GHG‑emission reduction targets [p. 175]; an investment policy based on the Greenhouse Gas Protocol that excludes fossil‑fuel, thermal‑coal and tobacco investments [p. 179]; an exclusion policy for its Responsible Investing funds that bars issuers involved in fossil fuels [p. 181]; and internal measures to reduce its own direct footprint (facilities and mobility) while offsetting remaining emissions to achieve net climate neutrality [p. 281].

Retrieved pages: [156, 175, 281, 179, 181]


## 6. Comparing models
To compare models fairly, retrieval runs once per question and every model receives exactly the same excerpts. Any difference in the answers is then due to the model, not to retrieval.

In [22]:
def compare_models(question, llms=LLM_MODELS, k=TOP_K):
    """Answer one question with each model, using identical retrieved chunks."""
    retrieved = retrieve(question, k=k)
    messages = build_messages(question, retrieved)

    rows = []
    for name in llms:
        rows.append({"model": name, "answer": call_llm(messages, llm=name)})
        time.sleep(PAUSE_SECONDS)

    print(f"Q: {question}")
    print(f"Retrieved pages: {retrieved['page'].tolist()}")
    return pd.DataFrame(rows)

### 6.1 A question the report answers

In [23]:
with pd.option_context("display.max_colwidth", None):
    display(compare_models("How does KBC take action to prevent child labour?"))

Q: How does KBC take action to prevent child labour?
Retrieved pages: [217, 153, 123, 236, 52]


,model,answer
0,nemotron,"KBC strictly adheres to all relevant legislation that prohibits child labour, treats the practice as unequivocally illegal, and ensures compliance across all its operations throughout the group【p. 217】."
1,apodex,"KBC strictly adheres to all relevant legislation, stating that child labour is unequivocally prohibited by law, and is fully committed to ensuring compliance across all operations throughout the group [p. 217]. Additionally, KBC integrates human rights in its due diligence processes and applies zero tolerance for severe human rights impacts, blacklisting the worst offenders of the UN Global Compact Principles [p. 236]."


**Observation:** Both models cite page 217. Nemotron gives a single-sentence answer, while Apodex adds detail on KBC's human rights due diligence process from page 236.

### 6.2 A question the report only loosely relates to

In [24]:
with pd.option_context("display.max_colwidth", None):
    display(compare_models("How does KBC celebrate festivals?"))

Q: How does KBC celebrate festivals?
Retrieved pages: [52, 61, 216, 22, 118]


,model,answer
0,nemotron,"KBC celebrates festivals by acting as the main sponsor of the largest sports festival in Slovakia (through ČSOB) and by participating in initiatives such as the “Good Deed Bank – Month of the Blood Donors,” whose opening event was held at K&H’s head office【p. 61】."
1,apodex,The provided excerpts do not contain this information.


**Observation:** Nemotron answered by pointing to ČSOB's sponsorship of a sports festival (p. 61), while Apodex replied that the excerpts do not contain the information. Since retrieval was identical, the difference comes from generation: one model stretched a loosely related fact to fit the question, the other declined. Nemotron also formatted its citation as 【p. 61】 rather than in the [p. N] format the prompt asks for, which matters if citations are later checked automatically.

### 6.3 A question the report cannot answer (negative test)

In [25]:
with pd.option_context("display.max_colwidth", None):
    display(compare_models("What is KBC's standing on the shape of earth - is it round or flat??"))

Q: What is KBC's standing on the shape of earth - is it round or flat??
Retrieved pages: [103, 349, 348, 353, 352]


,model,answer
0,nemotron,The provided excerpts do not contain this information.
1,apodex,The provided excerpts do not contain this information.


**Observation:** Both models returned the fixed refusal sentence, as instructed.

## 7. Known limitations and next steps
- **Repeated page footers** (e.g. "22 Annual Report KBC Group 2025") are not yet removed and remain inside many chunks.
- **Some text is extracted backwards**, likely from rotated table text (e.g. "stessa RAG latoT" for "Total GAR assets").
- **Some layout-heavy pages extract as garbled text** (e.g. page 2), which the length filter does not catch.
- **Chunks do not cross page boundaries**, so a paragraph split across two pages becomes two chunks.
- **Evaluation is qualitative so far.** The next step is a small labelled test set (question, expected page, expected answer) to measure retrieval hit rate and answer correctness for each model.
- **Data governance:** Embeddings are computed locally, and the full report never leaves the machine. For each question, only the question and the top retrieved excerpts (5 chunks of a public document) are sent to the free LLM API. Internal or confidential data would require a self-hosted or contractually covered model.